# CNN — Convolutional Neural Network

CNNs are built for spatial data (images, audio). Instead of connecting every pixel to every neuron (which would be billions of parameters), a Conv layer slides a small **filter** over the image and detects local patterns — edges, textures, shapes.

**Key idea:** the same filter is reused everywhere in the image (weight sharing) → fewer parameters, translation invariance.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# --- Understand Conv2d ---

# Conv2d(in_channels, out_channels, kernel_size)
# in_channels  = number of input feature maps (3 for RGB, 1 for grayscale)
# out_channels = number of filters (= number of output feature maps)
# kernel_size  = filter size (3 = 3×3 filter)

conv = nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1)

img_batch = torch.randn(8, 1, 28, 28)   # [batch, channels, height, width]
out = conv(img_batch)
print('Input shape: ', img_batch.shape)    # [8, 1, 28, 28]
print('Output shape:', out.shape)           # [8, 16, 28, 28] — 16 feature maps

# MaxPool2d — downsample by taking max in each region
pool = nn.MaxPool2d(kernel_size=2)
print('After pool:  ', pool(out).shape)    # [8, 16, 14, 14] — halved spatial dims

In [ ]:
# --- CNN Model for MNIST ---

class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv_block1 = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),  # [b, 32, 28, 28]
            nn.ReLU(),
            nn.MaxPool2d(2)                              # [b, 32, 14, 14]
        )
        self.conv_block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1), # [b, 64, 14, 14]
            nn.ReLU(),
            nn.MaxPool2d(2)                              # [b, 64, 7, 7]
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),             # [b, 64*7*7] = [b, 3136]
            nn.Linear(3136, 128),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(128, 10)        # 10 classes
        )

    def forward(self, x):
        x = self.conv_block1(x)
        x = self.conv_block2(x)
        return self.classifier(x)


device = 'cuda' if torch.cuda.is_available() else 'cpu'
model  = CNN().to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f'Device:     {device}')
print(f'Parameters: {total_params:,}')   # much fewer than MLP despite better accuracy

# Verify shapes with a dummy forward pass
dummy = torch.randn(2, 1, 28, 28).to(device)
print(f'Output shape: {model(dummy).shape}')   # [2, 10]

In [ ]:
# --- Data + Training ---
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_data   = datasets.MNIST('./data', train=True,  download=True, transform=transform)
test_data    = datasets.MNIST('./data', train=False, download=True, transform=transform)
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_data,  batch_size=64, shuffle=False)

loss_fn   = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(3):
    # Train
    model.train()
    correct = 0
    for X, y in train_loader:
        X, y  = X.to(device), y.to(device)
        loss  = loss_fn(model(X), y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        correct += (model(X).argmax(1) == y).sum().item()
    train_acc = correct / len(train_data)

    # Eval
    model.eval()
    correct = 0
    with torch.no_grad():
        for X, y in test_loader:
            X, y = X.to(device), y.to(device)
            correct += (model(X).argmax(1) == y).sum().item()
    test_acc = correct / len(test_data)

    print(f'Epoch {epoch+1} | train acc: {train_acc:.3f} | test acc: {test_acc:.3f}')